# Exceptions and error handling

This module covers Python's exception model: what exceptions are, how to catch them, how to raise them, and the patterns that make a tm1py script handle real failures rather than mask them. Readers may already be familiar with exceptions from another language or be meeting them for the first time; the focus here is on what Python's mechanism actually does and the idioms that have become standard around it.

Exceptions are how Python signals that something has gone wrong: a file is missing, a network call has failed, a key is not in a dict, a TM1 server has returned an error response. They are not an exotic feature reserved for hardened production code; almost every nontrivial script meets them within the first few lines. The question is not whether to deal with exceptions but where, and the wrong place is almost always "everywhere, with a bare `except`." This module covers the right places.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. What an exception is and the problem it solves
2. The try/except statement
3. Catching by exception type
4. The exception object
5. The exception hierarchy
6. Raising exceptions
7. Reraising and exception chaining
8. The else and finally clauses
9. Custom exception classes
10. EAFP vs LBYL
11. Context managers as cleanup
12. ExceptionGroup and except\*
13. assert and traceback
14. Real world design principles
15. Common mistakes

---

## 1. What an exception is and the problem it solves

A tm1py script that reads a config file and connects to a TM1 server has many ways to fail. The config file might not exist. The file might exist but contain malformed JSON. The TM1 server might be unreachable. The username might be wrong. The cube the script wants to write to might not exist. Each of these is something the author cannot prevent at write time; the failure happens at runtime, in production, often months after the script was written.

Without a dedicated mechanism, every function that can fail has to return both its result and a status code, and every caller has to check the status:

In [ ]:
config, error = read_config("config.json")
if error is not None:
    print(f"could not read config: {error}")
    return
tm1, error = connect(config)
if error is not None:
    print(f"could not connect: {error}")
    return
# ... three lines of error checking per line of real work

This is the C style approach. It works, but the error handling drowns out the logic, a missed check silently propagates the wrong value, and there is no way to enforce that callers handle errors at all.

Python's exception mechanism inverts the model. A function that fails **raises** an exception, and the call returns nothing. Control jumps out of the function and continues unwinding the call stack until it reaches code that **catches** the exception. The normal path is written without any error checking; the error path lives in one place, separately:

In [ ]:
try:
    config = read_config("config.json")
    tm1 = connect(config)
    cubes = tm1.cubes.get_all_names()
except FileNotFoundError as exc:
    print(f"config file missing: {exc}")
except ConnectionError as exc:
    print(f"could not reach TM1: {exc}")

The `try` block reads top to bottom as if nothing can fail; the `except` clauses describe what to do if it does. The next sections cover each part of this in detail.

## 2. The try/except statement

The `try` statement marks a block of code where an exception might be raised. One or more `except` clauses follow, each handling a specific kind of exception:

In [ ]:
import json
from pathlib import Path

try:
    text = Path("config.json").read_text(encoding="utf-8")
    config = json.loads(text)
except FileNotFoundError:
    config = {}                          # treat missing file as empty config

If the `try` body completes without raising, the `except` clauses are skipped entirely. If the body raises, Python looks at each `except` clause in order and runs the first one whose type matches the exception. If no clause matches, the exception keeps propagating out of the `try` statement and up the call stack.

Once an exception is caught, control continues at the statement after the entire `try`/`except`. The exception is considered handled; it does not reraise on its own. The rest of the script runs normally:

In [ ]:
try:
    text = Path("config.json").read_text(encoding="utf-8")
    config = json.loads(text)
except FileNotFoundError:
    config = {}

print("config loaded:", len(config), "keys")   # runs whether the file existed or not

A `try` with no `except` (and no `finally`, see Topic 8) is a syntax error. The statement must do something with the exception, even if that something is just `pass`.

## 3. Catching by exception type

Each `except` clause names the exception type it handles, or a tuple of types. Only matching exceptions trigger that clause; everything else propagates:

In [ ]:
import json
from pathlib import Path

try:
    text = Path("config.json").read_text(encoding="utf-8")
    config = json.loads(text)
except FileNotFoundError:
    config = {}                          # missing file becomes empty config
except json.JSONDecodeError:
    raise SystemExit("config.json is not valid JSON")

Multiple types in one clause are written as a tuple:

In [ ]:
try:
    response = tm1.cells.execute_view_values(cube_name="Sales Plan", view_name="Plan Input")
except (ConnectionError, TimeoutError):
    response = {}                        # network blip becomes empty response

The match uses `isinstance`, so an `except` for a base class catches subclasses too. `except OSError` catches `FileNotFoundError`, `PermissionError`, `IsADirectoryError`, and the dozen other I/O errors that inherit from it. This is usually a feature; it can also be a bug when the broad clause swallows something the narrow one should have handled. Order matters: Python checks `except` clauses top to bottom and uses the first match, so put more specific types before more general ones.

In [ ]:
# Wrong order
try:
    ...
except OSError:
    ...
except FileNotFoundError:                # never reached; OSError caught it first
    ...

# Correct order
try:
    ...
except FileNotFoundError:
    ...
except OSError:                          # all other OS errors
    ...

## 4. The exception object

The `as name` clause binds the caught exception to a local variable so its details can be inspected:

In [ ]:
import json
from pathlib import Path

try:
    text = Path("config.json").read_text(encoding="utf-8")
    config = json.loads(text)
except FileNotFoundError as exc:
    print(f"config not found at {exc.filename}")
except json.JSONDecodeError as exc:
    print(f"invalid JSON at line {exc.lineno}, column {exc.colno}: {exc.msg}")

Most exception classes carry attributes specific to the failure. `FileNotFoundError` has `filename` and `errno`. `json.JSONDecodeError` has `msg`, `doc`, `pos`, `lineno`, and `colno`. tm1py's `TM1pyRestException` has `status_code`, `reason`, and `headers`. The exception's `args` tuple holds whatever was passed to the constructor; `str(exc)` produces a human readable message.

The name bound by `as` is scoped to the `except` clause and unbound when the clause exits, even on success. This is a deliberate cleanup choice: a reference held in the exception keeps the entire traceback alive, which can pin large objects in memory. Code that needs the exception outside the clause must copy it explicitly:

In [ ]:
saved: Exception | None = None
try:
    risky_call()
except RuntimeError as exc:
    saved = exc                          # explicit copy outside the clause's scope

if saved is not None:
    log_failure(saved)

## 5. The exception hierarchy

Every exception in Python is an instance of a class, and those classes form a tree rooted at `BaseException`. The structure that matters for almost all code:

```
BaseException
 ├── SystemExit
 ├── KeyboardInterrupt
 ├── GeneratorExit
 └── Exception                ← almost everything inherits from here
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── OSError
      │    ├── FileNotFoundError
      │    ├── PermissionError
      │    └── ConnectionError
      ├── ValueError
      ├── TypeError
      └── ... (many more)
```

Two design points matter. First, ordinary "something went wrong in the program" exceptions inherit from `Exception`, not directly from `BaseException`. The siblings of `Exception` (`SystemExit`, `KeyboardInterrupt`, `GeneratorExit`) are control flow events that look like exceptions: `sys.exit()` raises `SystemExit`, hitting Ctrl+C raises `KeyboardInterrupt`, a generator being garbage collected raises `GeneratorExit`. None of them are bugs to be handled; they are signals to honor.

Second, the conventional broad catch is `except Exception`, never `except BaseException` and never bare `except`. `except Exception` catches programming and library errors but lets `KeyboardInterrupt` and `SystemExit` continue propagating, so the user can still cancel the script with Ctrl+C and the script can still call `sys.exit()` to terminate cleanly:

In [ ]:
try:
    do_the_long_load()
except Exception as exc:                 # not BaseException; not bare except
    log.exception("load failed")
    raise SystemExit(1)
# Ctrl+C still works inside do_the_long_load

tm1py's exceptions live in their own subtree under `Exception`: `TM1pyException` is the base, and `TM1pyRestException` (HTTP errors), `TM1pyRestrictionException`, and others inherit from it. The same `isinstance` rules apply: `except TM1pyException` catches all of them.

## 6. Raising exceptions

A function signals that something has gone wrong by **raising** an exception with `raise`:

In [ ]:
PERIODS: set[str] = {"Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"}

def get_period(period: str) -> str:
    if period not in PERIODS:
        raise ValueError(f"unknown period: {period!r}")
    return period

The argument to the exception class is the message. It becomes the exception's `args[0]` and what `str(exc)` returns. Make it specific: include the offending value, not just "invalid input."

Pick the most specific built in exception that fits. `ValueError` for an argument that is the right type but the wrong value. `TypeError` for an argument of the wrong type. `KeyError` for a lookup that should have succeeded. `LookupError` is the base class for both `KeyError` and `IndexError` and is rarely raised directly. `RuntimeError` is the catch all when nothing more specific applies; reach for it only when the choice is conscious, not lazy.

Two conventions worth following. First, the message goes in the exception, not in a `print` next to it. Anything that catches the exception sees the message; a `print` is invisible to the handler. Second, do not raise inside a normal control flow path. Exceptions are for situations the caller did not expect; if a function "raises" `KeyError` to mean "key not in cache, look it up elsewhere", every caller has to wrap it in a `try`/`except`, which is the C return code pattern with extra indentation. Use a sentinel value or `Optional` for the expected absence case; raise only for the unexpected.

## 7. Reraising and exception chaining

A handler that catches an exception, does something with it, and wants the exception to keep propagating uses bare `raise`:

In [ ]:
try:
    tm1.cells.write_values("Sales Plan", cells)
except TM1pyRestException as exc:
    log.error("write failed: status %s, reason %s", exc.status_code, exc.reason)
    raise                                # reraise the same exception, with its original traceback

Bare `raise` inside an `except` block reraises the currently active exception. The traceback is preserved, so the caller still sees where the original error happened. The shorter `raise exc` form also reraises but rewrites the traceback to point at the current line, which makes the original location harder to find in logs; the bare form is the one to use.

Sometimes a handler wants to translate one exception into another that fits the abstraction better. The `raise NewException(...) from exc` form attaches the original to the new one as the **cause**:

In [ ]:
try:
    config = json.loads(text)
except json.JSONDecodeError as exc:
    raise ConfigError(f"could not parse config: {exc.msg}") from exc

The traceback printed by Python now reads "the above exception was the direct cause of the following exception", which keeps the trail visible while letting callers catch a clean `ConfigError` without knowing about JSON internals. Without `from exc`, Python attaches the original as the **context** instead and prints "during handling of the above exception, another exception occurred" — useful but accidental looking. Use `from exc` to mark the link as intentional, or `from None` to suppress the chain entirely when the underlying exception is genuinely irrelevant to callers.

## 8. The else and finally clauses

A `try` statement can carry up to two more clauses after `except`. The `else` clause runs only if the body completed without raising:

In [ ]:
try:
    text = Path("config.json").read_text(encoding="utf-8")
except FileNotFoundError:
    config = {}
else:
    config = json.loads(text)            # runs only on the no exception path

The point of `else` is to keep the `try` body as small as possible. Code that should not be guarded by the `except` clauses goes in `else`, not at the end of the `try`. If `json.loads` raises a `JSONDecodeError`, that error should propagate; putting the call in `else` makes sure the `FileNotFoundError` clause does not accidentally catch a parse failure that happens to throw a related type.

The `finally` clause runs after everything else, on every exit path: success, caught exception, uncaught exception, even `return` from inside the `try`. It is the right place for cleanup that must happen no matter what:

In [ ]:
tm1 = TM1Service(...)
try:
    tm1.cells.write_values("Sales Plan", cells)
finally:
    tm1.logout()                         # runs on success and on exception

`finally` does not swallow the exception: any exception from the `try` body keeps propagating after `finally` runs. The only exception is when `finally` itself uses `return` or raises a new exception, which then replaces the original — usually a bug. Topic 11 covers the cleaner alternative with context managers.

## 9. Custom exception classes

A project usually defines its own exception classes for its own failure modes. Custom classes serve two purposes: they describe the error in the project's own vocabulary, and they let callers catch the project's failures specifically without catching unrelated `ValueError`s from a stdlib call.

In [ ]:
class LoadError(Exception):
    """Base class for all errors raised by the cube load script."""

class ConfigError(LoadError):
    """The config file is missing, unreadable, or invalid."""

class SourceUnavailable(LoadError):
    """The source database could not be reached."""

class CubeWriteFailed(LoadError):
    """tm1py reported a failure writing cells into the cube."""

Three conventions worth following. First, derive from `Exception` (or a project specific subclass of it), never from `BaseException`. Second, give the project a single root exception (`LoadError` here) so callers can catch every error from the module with one `except LoadError`. Third, do not add behavior to the class unless needed: a docstring or a one line `pass` body is enough. The class name and the message in the constructor carry almost all the meaning.

For exceptions that need to carry structured data, override `__init__` and store fields:

In [ ]:
class CubeWriteFailed(LoadError):
    def __init__(self, cube: str, count: int, status: int) -> None:
        super().__init__(f"writing {count} cells to {cube!r} failed with status {status}")
        self.cube = cube
        self.count = count
        self.status = status

Callers can now branch on `exc.status` or log `exc.cube` without parsing the message string. The string form, set via `super().__init__`, stays human readable for the cases that just want to print.

## 10. EAFP vs LBYL

Two approaches to handling things that might fail. **Look Before You Leap** (LBYL) checks whether the operation will succeed before attempting it:

In [ ]:
import os

if os.path.exists("config.json"):
    text = open("config.json").read()
else:
    text = ""

**Easier to Ask Forgiveness than Permission** (EAFP) attempts the operation and handles the failure:

In [ ]:
try:
    text = open("config.json").read()
except FileNotFoundError:
    text = ""

Python idiomatically prefers EAFP for three reasons. First, LBYL has a race condition: between `os.path.exists` and `open`, another process could delete the file. EAFP has no race condition; the file either opens or it does not. Second, LBYL duplicates work the operation already does internally, since `open` checks for existence anyway. Third, LBYL forks the code into "check" and "do" branches that drift apart over time; EAFP keeps them together.

EAFP is not a license to wrap everything in `try`/`except`. The rule is to use it where the failure is genuinely exceptional and non local, and use a normal conditional where the test is cheap and the absence is part of normal flow:

In [ ]:
# EAFP: the key is required; absence is unusual and signals a bad config
try:
    period = config["period"]
except KeyError:
    raise ConfigError("config is missing required 'period' key")

# Conditional: the key is optional; both presence and absence are normal
period = config.get("period", "Jan")

For dict access, `.get()` with a default is the idiomatic conditional. For attribute access, `getattr(obj, "name", default)` is the parallel. Reach for `try`/`except` when there is no such helper or when the operation is too complex to test cheaply.

## 11. Context managers as cleanup

Topic 8 introduced `finally` for cleanup. In practice, almost no modern Python code uses `try`/`finally` directly; it uses the `with` statement instead. `with` is `try`/`finally` with the cleanup attached to the resource itself, so the call site does not have to remember it (covered in detail in the context managers module).

In [ ]:
# try/finally form
tm1 = TM1Service(...)
try:
    tm1.cells.write_values("Sales Plan", cells)
finally:
    tm1.logout()

# with form, equivalent and preferred
with TM1Service(...) as tm1:
    tm1.cells.write_values("Sales Plan", cells)

The two compose: a `with` statement can contain a `try`/`except` for handling errors, and the cleanup still runs on every exit path:

In [ ]:
with TM1Service(...) as tm1:
    try:
        tm1.cells.write_values("Sales Plan", cells)
    except TM1pyRestException as exc:
        log.error("write failed: %s", exc.reason)
        raise CubeWriteFailed("Sales Plan", len(cells), exc.status_code) from exc
# tm1.logout() runs whether the write succeeded or raised

`contextlib.suppress` covers the narrow case of "ignore one specific exception":

In [ ]:
from contextlib import suppress
from pathlib import Path

with suppress(FileNotFoundError):
    Path("stale_lock").unlink()          # delete if present, do nothing if absent

The `with suppress(...)` form reads as a single intent and is shorter than `try: ... except FileNotFoundError: pass`. It catches only the listed exception types; everything else propagates.

## 12. ExceptionGroup and except\*

Some operations can fail in more than one way at the same time. An `asyncio.gather` of three concurrent tm1py calls might see one connection refused, one timeout, and one succeed — three independent results that need to be reported together. Python 3.11 added **exception groups** for this: a single `ExceptionGroup` that wraps a list of inner exceptions.

In [ ]:
errors = [
    ConnectionError("tm1-emea unreachable"),
    TimeoutError("tm1-amer timeout after 30s"),
]
raise ExceptionGroup("multiple servers failed", errors)

A handler unpacks the group with `except*` (note the asterisk), which matches each inner exception against the type and lets the rest reraise:

In [ ]:
try:
    fetch_from_all_servers()
except* ConnectionError as eg:
    log.error("connection errors: %s", [str(e) for e in eg.exceptions])
except* TimeoutError as eg:
    log.error("timeouts: %s", [str(e) for e in eg.exceptions])

Each `except*` clause runs at most once per `try` and receives a subgroup containing only the matching exceptions; the rest are reraised in their own group. This is a Python 3.11+ feature; on older versions, libraries that need similar behavior carry a list of exceptions inside a single class. Most tm1py scripts will not see `ExceptionGroup` unless they use `asyncio.TaskGroup` or a library that emits it.

## 13. assert and traceback

Two adjacent tools complete the picture.

`assert` raises `AssertionError` if its condition is false. It is for **invariants** the developer believes are always true at a given point — sanity checks that catch bugs, not error handling for things the user might do wrong:

In [ ]:
def write_cells(cube: str, cells: dict[tuple[str, ...], float]) -> None:
    assert all(isinstance(v, (int, float)) for v in cells.values()), \
        "all cell values must be numeric"
    tm1.cells.write_values(cube, cells)

Two warnings. First, `assert` is stripped when Python is run with the `-O` (optimize) flag; never put logic that must run inside an `assert`, only checks. Second, use it for "this should never happen, and if it does the program is broken", not for validating user input — for that, raise `ValueError` explicitly.

The `traceback` module formats the same traceback Python prints on an uncaught exception. The most common use is logging an exception in a long running service so the operator sees what failed without crashing the process:

In [ ]:
import logging
import traceback

log = logging.getLogger(__name__)

try:
    do_the_load()
except Exception:
    log.exception("load failed")         # shorthand: includes traceback automatically
    # or, equivalently:
    # log.error("load failed:\n%s", traceback.format_exc())

`log.exception(msg)` is `log.error(msg, exc_info=True)` and is the right form inside an `except` block: it captures the current exception's traceback automatically. `traceback.format_exc()` is the lower level form that returns the traceback as a string for cases where the logger is not available.

## 14. Real world design principles

**Catch `Exception`, not `BaseException`, and never bare `except`.** A bare `except` swallows `KeyboardInterrupt` and `SystemExit`, breaking Ctrl+C and `sys.exit()`. The conventional broad catch is `except Exception`, which covers everything an ordinary failure could be without trapping control flow signals.

**Catch the narrowest exception that fits.** `except FileNotFoundError` is better than `except OSError`, which is better than `except Exception`. The narrower the catch, the less the handler can mask. Reach for `except Exception` only at the outermost layer of a script or service, where the goal is "log and exit cleanly no matter what."

**Log exceptions where they happen, decide where to act.** A library function should rarely catch and swallow; it should let exceptions propagate to the caller, who has the context to decide what to do. Logging belongs at the boundary where the script meets its operator (the main function, the request handler, the cron entry point), not at every level in between.

**Translate exceptions at module boundaries.** A function that calls `json.loads` and `requests.get` internally should not let `JSONDecodeError` and `requests.ConnectionError` leak to its callers; that exposes implementation details. Catch them inside the function and reraise as a project specific exception with `raise ConfigError(...) from exc`. The chained traceback preserves the original for debugging.

**Use `with`, not `try/finally`, for cleanup.** Anything that needs `try/finally` is a candidate for a context manager. Pushing the cleanup into the resource means every caller gets it for free; leaving it as `try/finally` means every caller has to remember it.

**Do not use exceptions for control flow.** Raising an exception just to break out of a function is clever and unreadable. Exceptions are a real cost (raising and catching one is hundreds of nanoseconds, vs single nanoseconds for a comparison), and more importantly they change the meaning of the code. If the situation is part of normal flow, return a value or use a conditional.

**Reraise with `raise`, not `raise exc`.** A bare `raise` inside an `except` clause preserves the original traceback. `raise exc` reraises but reattaches the traceback to the current line, making the original location harder to find in logs.

**Validate user input with explicit raises.** A function that accepts arguments from outside the system should check them and raise `ValueError` or `TypeError` with a clear message. `assert` is for internal invariants; user input is not an invariant, it is exactly what cannot be trusted.

## 15. Common mistakes

**Bare `except` catching everything.**

In [ ]:
# Wrong
try:
    do_the_load()
except:                                  # catches KeyboardInterrupt, SystemExit, everything
    print("something went wrong")

# Correct
try:
    do_the_load()
except Exception:
    log.exception("load failed")

**Catching too broadly and hiding bugs.**

In [ ]:
# Wrong
try:
    config = json.loads(text)
    cube = config["cube"]
    rows = tm1.cells.execute_view_values(cube, "Plan Input")
except Exception:
    rows = {}                            # hides KeyError, ConnectionError, JSONDecodeError, anything

# Correct
try:
    rows = tm1.cells.execute_view_values(cube, "Plan Input")
except TM1pyRestException as exc:
    if exc.status_code == 404:
        rows = {}
    else:
        raise

**`raise exc` instead of bare `raise`.**

In [ ]:
# Wrong
try:
    risky()
except SomeError as exc:
    log.error("risky failed")
    raise exc                            # reraises, but traceback now points here

# Correct
try:
    risky()
except SomeError:
    log.error("risky failed")
    raise                                # original traceback preserved

**Using `assert` for runtime validation.**

In [ ]:
# Wrong
def get_period(period):
    assert period in PERIODS             # stripped under python -O
    return period

# Correct
def get_period(period):
    if period not in PERIODS:
        raise ValueError(f"unknown period: {period!r}")
    return period

**Translating an exception without `from`.**

In [ ]:
# Wrong
try:
    config = json.loads(text)
except json.JSONDecodeError as exc:
    raise ConfigError(f"bad config: {exc}")        # original attached as accidental, not cause

# Correct
try:
    config = json.loads(text)
except json.JSONDecodeError as exc:
    raise ConfigError(f"bad config: {exc}") from exc

**`try/finally` instead of `with` for resource cleanup.**

In [ ]:
# Wrong
tm1 = TM1Service(...)
try:
    tm1.cells.write_values("Sales Plan", cells)
finally:
    tm1.logout()

# Correct
with TM1Service(...) as tm1:
    tm1.cells.write_values("Sales Plan", cells)

**Using exceptions for normal control flow.**

In [ ]:
# Wrong
try:
    period = config["period"]
except KeyError:
    period = "Jan"                       # the absence is normal; no need to raise/catch

# Correct
period = config.get("period", "Jan")

**Catching the broad clause before the narrow one.**

In [ ]:
# Wrong
try:
    open(path).read()
except OSError:                          # catches FileNotFoundError too
    handle_other_os_error()
except FileNotFoundError:                # never reached
    handle_missing()

# Correct
try:
    open(path).read()
except FileNotFoundError:
    handle_missing()
except OSError:
    handle_other_os_error()